In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path("/home/mersad/protBuild")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [2]:
import os
import json
import torch
import multiprocessing as mp
from torch.utils.data import DataLoader
from ml.src.training.dataset.protein_dataset import ProteinDataset
from ml.src.models.tokenizer.bpe import BPETokenizer
from ml.src.models.gpt2.gpt2 import GPT2
from ml.src.training.trainer.train import train_model
from torch.optim.lr_scheduler import CosineAnnealingLR


In [3]:
config = {}
config_path = PROJECT_ROOT / 'ml/src/models/configs/gpt2-config.json'
selected_config = 'small'
with open (config_path, 'r') as f:
    data = json.load(f)
    if selected_config in data:
        config = data[selected_config]
    else:
        raise KeyError(f"Configuration '{selected_config}' not found.")


In [4]:
vocab_path = PROJECT_ROOT /'ml/datasets/converted/sequences/tokenizer/proteinas_bpe_1200_vocab.json'
merges_path = PROJECT_ROOT /'ml/datasets/converted/sequences/tokenizer/proteinas_bpe_1200_merges.json'
train_corpus_path = PROJECT_ROOT /'ml/datasets/converted/sequences/proteinas_train.txt'
val_corpus_path = PROJECT_ROOT /'ml/datasets/converted/sequences/proteinas_test.txt'
end_token='<|endofprotein|>'
checkpoint_path=PROJECT_ROOT /'ml/notebooks/checkpoints/'
save_file_name='checkpoint-model.pth'
device = 'cuda' if torch.cuda.is_available() else 'cpu'
tokenizer = BPETokenizer().load_vocab_and_merges(vocab_path, merges_path)


In [5]:
torch.manual_seed(123)

In [6]:
num_workers = 1
train_position_array = mp.Array('q', num_workers)
train_idx_array = mp.Array('q', num_workers)

resume_state = {}
if os.path.exists(os.path.join(checkpoint_path, save_file_name)):
    checkpoint = torch.load(os.path.join(checkpoint_path, save_file_name), map_location='cpu', weights_only=False)
    saved_offset = checkpoint.get('worker_byte_offsets', {})
    saved_idxs = checkpoint.get('worker_next_idx', {})
    resume_state = {
        w: (saved_offset.get(w, 0), saved_idxs.get(w, 0))
        for w in range(num_workers)
    }

In [7]:
train_data = ProteinDataset(
    corpus_path=train_corpus_path, tokenizer=tokenizer,
    context_length=config['context_length'], end_token=end_token,
    resume_state=resume_state, position_array=train_position_array,
    idx_array=train_idx_array
)
train_eval_dataset = ProteinDataset(
    corpus_path=train_corpus_path,
    tokenizer=tokenizer,
    context_length=config['context_length'],
    end_token="<|endofprotein|>",
    buffer_size=1,
)
val_data = ProteinDataset(corpus_path=val_corpus_path, tokenizer=tokenizer,
                         context_length=config['context_length'], end_token=end_token)

In [8]:
train_loader = DataLoader(
    train_data,
    batch_size=2,
    num_workers=num_workers,
    pin_memory=True,
)

train_eval_loader = DataLoader(
    train_eval_dataset,
    batch_size=2,
    num_workers=0,
)

val_loader = DataLoader(
    val_data,
    batch_size=2,
    num_workers=num_workers,
    pin_memory=True,
)

In [9]:
gpt2_model = GPT2(
    emb_dim=config['emb_dim'],
    d_out=config['emb_dim'],
    vocab_size=config['vocab_size'],
    context_length=config['context_length'],
    num_heads=config['n_heads'],
    n_layers=config['n_layers'],
    dropout=config['drop_rate'],
    qkv_bias=config['qkv_bias']
)

In [10]:
from torch.optim.lr_scheduler import LinearLR, CosineAnnealingLR, SequentialLR
optimizer = torch.optim.AdamW(gpt2_model.parameters(), lr=3e-4, weight_decay=0.1)

warmup_steps = 500
total_steps = 500_000

warmup_scheduler = LinearLR(
    optimizer,
    start_factor=0.01,
    end_factor=1.0,
    total_iters=warmup_steps
)

cosine_scheduler = CosineAnnealingLR(
    optimizer,
    T_max=total_steps - warmup_steps,
    eta_min=1e-5
)

scheduler = SequentialLR(
    optimizer,
    schedulers=[warmup_scheduler, cosine_scheduler],
    milestones=[warmup_steps]
)


In [11]:
train_losses, val_losses, track_tokens_seen = train_model(gpt2_model, train_loader, val_loader, train_eval_loader,
             optimizer, scheduler, device,15, 250, 200, train_position_array, train_idx_array, num_workers, checkpoint_path, save_file_name)

Epoch 1/15: 0batch [00:03, ?batch/s, loss=7.274]

Epoch 1 (Step 0):
Train loss 7.260, Val loss 7.263
Train perplexity 1421.634, Val perplexity 1426.230


Epoch 1/15: 250batch [00:48,  9.06batch/s, loss=6.092]

Epoch 1 (Step 250):
Train loss 6.057, Val loss 6.064
Train perplexity 427.156, Val perplexity 429.906


Epoch 1/15: 500batch [01:34,  8.99batch/s, loss=6.020]

...

Epoch 15/15: 1020batch [03:13,  8.78batch/s, loss=4.898]

Epoch 15 (Step 21250):
Train loss 4.292, Val loss 6.478
Train perplexity 73.122, Val perplexity 650.863


Epoch 15/15: 1270batch [04:00,  8.76batch/s, loss=4.843]

Epoch 15 (Step 21500):
Train loss 4.395, Val loss 6.472
Train perplexity 81.014, Val perplexity 646.682


Epoch 15/15: 1445batch [04:38,  5.18batch/s, loss=4.919]


In [12]:
torch.save(gpt2_model.state_dict(), 'protbuildv1.pth')

In [13]:
from ml.src.inference.sequence_generator import generate_protein

In [14]:
state_dict = torch.load("protbuildv1.pth", weights_only=True)
gpt2_model.load_state_dict(state_dict)
gpt2_model = gpt2_model.to(device)
gpt2_model.eval()

GPT2(
  (tok_emb): Embedding(1200, 512)
  (pos_emb): Embedding(512, 512)
  (dropout): Dropout(p=0.1, inplace=False)
  (transformer_blocks): Sequential(
    (0): TransformerBlock(
      (norm1): Normalization()
      (attention): MultiHeadAttention(
        (Wquery): Linear(in_features=512, out_features=512, bias=True)
        (Wkey): Linear(in_features=512, out_features=512, bias=True)
        (Wvalue): Linear(in_features=512, out_features=512, bias=True)
        (proj): Linear(in_features=512, out_features=512, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (dropout): Dropout(p=0.1, inplace=False)
      (norm2): Normalization()
      (ff): FeedForward(
        (layers): Sequential(
          (0): Linear(in_features=512, out_features=2048, bias=True)
          (1): GELU()
          (2): Linear(in_features=2048, out_features=512, bias=True)
        )
      )
    )
    (1): TransformerBlock(
      (norm1): Normalization()
      (attention): MultiHeadAttention(


In [15]:
prompts = [
    "MKWVTFISLLLLFSSAYSRGVFRR",
    "MKTIIALSYIFCLVFAD",
    "MVLSPADKTNVKAAWGKVGA"
]

for prompt in prompts:
    print("=" * 80)
    print(prompt)

    output = generate_protein(
        gpt2_model,
        prompt=prompt,
        tokenizer=tokenizer,
        max_new_tokens=100,
        context_size=config["context_length"],
        device=device,
        temperature=1.0,
        topk=25
    )

    print(output)

MKWVTFISLLLLFSSAYSRGVFRR
MKWVTFISLLLLFSSAYSRGVFRREPTDKDVYNSIHKFTQKRKNGKFNIYMTTQFMLNGDHRVGMLRKCYWCTHAKTQTHAMVYIHSSPCNEMSVTAQLLTLMDIPIYGVIAHMVFTVDEMKIPNDNYESC<|endofprotein|>
MKTIIALSYIFCLVFAD
MKTIIALSYIFCLVFADRAKMKDHKLWMNFSKSVECAQAHETIMKTHLDRSCKTASLFPLEGHCKICGMEQFMHAQHYDCMDFQQGFGCGGLQGKYTRGEIP<|endofprotein|>
MVLSPADKTNVKAAWGKVGA
MVLSPADKTNVKAAWGKVGAFTGIYRMPDENAIPGCKTRYNRCWGSEGFAAIGNRGYTTKWEAGL<|endofprotein|>


In [16]:
temperatures = [0, 0.5, 0.8, 1.0, 1.2]

for temperature in temperatures:
    output = generate_protein(
        gpt2_model,
        prompt="MVLSPADKTNVKAAWGKVGA",
        tokenizer=tokenizer,
        max_new_tokens=100,
        context_size=config["context_length"],
        device=device,
        temperature=temperature,
        topk=25
    )

    print(f"\nTemperature = {temperature}")
    print(output)


Temperature = 0
MVLSPADKTNVKAAWGKVGADHQHTIPHIQSQTKMTKEDHWDKCTQPYISCPYIMKHPIHRIMMTIMKFNFCYVYTVTTWIDFTQ<|endofprotein|>

Temperature = 0.5
MVLSPADKTNVKAAWGKVGADHLLCNEQNAQKDYIYKSYNHYYIKAVRRCGWPHFHKMMTKTIFMTYVANWWVL<|endofprotein|>

Temperature = 0.8
MVLSPADKTNVKAAWGKVGADHPTTAPIVAPFFWKANSMIAMPFCMHGNCGVISFNSAHETCGLMSVIDRIHVRWTWNDPLWGFQEDDHADWACFDWWGQETDSKWAHFFKKQTYRIQ<|endofprotein|>

Temperature = 1.0
MVLSPADKTNVKAAWGKVGAYNHEYNSAQGNDGNIDFMNINHFMFRVNCWLLSNTIHMTDFSEVLPETP<|endofprotein|>

Temperature = 1.2
MVLSPADKTNVKAAWGKVGANYTCKYYDYIWIIYMRRIMNAILSAKYMPHDQYQHQSECNTWAIGVYYFHDVFIPPRTEYVDVDNRRMCGSPFWGSQPEFNEHEWIDVFCA<|endofprotein|>
